In [ ]:
import base64
import os
import pandas as pd
from openai import OpenAI

# Set OPENAI_API_KEY as an environment variable before running, e.g.
#   export OPENAI_API_KEY=sk-...
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


features_raw = pd.read_csv(os.environ.get("FEATURES_RAW_CSV", "../../dataset/features_raw_251001.csv"))
features_norm = pd.read_csv(os.environ.get("AUDIO_CLUSTER_CSV", "../../dataset/audio_cluster_54_k5_v1.csv"))
audio_info = features_norm[["keycode", "kmeans_cluster", "mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0", "F1", "F2",
                        "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio",
                        "arousal", "valence", "std_arousal", "std_valence"]]

columns_to_suffix = ["mean_f0", "std_f0", "VUV", "speaking_rate_w", "MFCC0", "F1", "F2",
                        "falling_ratio", "rising_ratio", "rising-falling_ratio", "falling-rising_ratio",
                        "arousal", "valence", "std_arousal", "std_valence"]
suffix = "_norm"

df = audio_info.rename(columns={col: col + suffix for col in audio_info.columns if col in columns_to_suffix})

combined = features_raw.merge(df, on="keycode",how="inner")
summary = combined.groupby("kmeans_cluster").mean(columns_to_suffix).reset_index()

cluster_id = 0
row = summary.loc[cluster_id, ["mean_f0_norm", "std_f0_norm", "speaking_rate_w_norm", "VUV_norm", "F1_norm", "F2_norm", "MFCC0_norm",
                    "arousal_norm", "valence_norm", "std_arousal_norm", "std_valence_norm"]]

f0, f0_std, sr, VUV, f1, f2, MFCC, a, v, a_std, v_std = row

img = client.images.generate(
    model="gpt-image-1.5",
    prompt="Generate a front-facing headshot portrait of a middle-aged(around 60 years old) female senator against a plain, neutral background.\
        The portrait must depict the same individual, with consistent facial identity, hairstyle, and lighting.\
        The aspect ratio of each portrait should be 1:1 (square format).\
        I have seven groups, and the following values are z-score transformed, \
        generate a facial portrait with AU01 {AU01}, AU02 {AU02}, AU04 {AU04}, \
        AU06 {AU06}, AU12 {AU12}, AU15 {AU15}, Au20 {AU20}, AU25 {AU25} \ ",
    n=1,
    size="1024x1024"
)

image_bytes = base64.b64decode(img.data[0].b64_json)
with open("output.png", "wb") as f:
    f.write(image_bytes)

In [ ]:
import base64
import os
from openai import OpenAI

# Set OPENAI_API_KEY as an environment variable before running.
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

img = client.images.generate(
    model="gpt-image-1.5",
    prompt="Generate a front-facing headshot portrait of a middle-aged(around 60 years old) female senator against a plain, neutral background.\
        The portrait must depict the same individual, with consistent facial identity, hairstyle, and lighting.\
        The aspect ratio of each portrait should be 1:1 (square format).\
        I have seven groups, and the following values are z-score transformed, \
        generate a facial portrait with AU01: 0.47, AU02: 0.31, AU04: 0.37, \
        AU06: 0.16, AU12: 0.13, AU15: 0.50, AU20: 0.26, AU25: 0.53 ",
    n=1,
    size="1024x1024"
)

image_bytes = base64.b64decode(img.data[0].b64_json)
with open("output.png", "wb") as f:
    f.write(image_bytes)